# Look at FCTD

In [ ]:
from nescafe23_fluctuations import tools
import xarray as xr
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv, find_dotenv
from scipy import io
import cmps
import seaborn as sns
import numpy as np
import pygenray as pr
import pickle

load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/ray_geo_sweep.pkl"

with open(fn, 'rb') as f:
    sweep = pickle.load(f)

### get lower array depths from pressure sensors

In [ ]:
dss = tools.open_receptions()

In [ ]:
lower_array_depth = []
for k in range(23):
    lower_array_depth.append(float(dss[k].actual_depth[32:].mean()))

lower_array_depth = np.array(lower_array_depth)

In [ ]:
Z_lower_array = np.mean(lower_array_depth)

In [ ]:
Z_lower_array

In [ ]:
fn = '../auxiliary/mean_pulse_lower.nc'
mean_pulse_lower = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_upper.nc'
mean_pulse_upper = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_lower_noise.nc'
mean_pulse_lower_noise = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_upper_noise.nc'
mean_pulse_upper_noise = xr.open_dataarray(fn)

### Sweep source depth and range values

In [ ]:
# previously used 550z 10000r
fctd = tools.compute_filtered_fctd_grid(z_smooth=500, r_smooth=10000)

# shift front location
front_location = 35e3  # m

fctd = fctd.assign_coords({
    "range": (fctd.range.values + front_location)
})

c = fctd["c"]

bathy = xr.DataArray(
    np.linspace(4908, 4908, 100),  # flat bottom
    dims=["range"],
    coords={"range": np.linspace(0, 200e3, 100)},
)

envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)

In [ ]:
source_depths = np.array([2000, 2050, 2100, 2150, 2200, 2250, 2300]).astype(float)
receiver_ranges = np.array([148e3, 149e3, 150e3, 151e3, 152e3]).astype(float)

ssource_depths, rreceiver_ranges = np.meshgrid(source_depths, receiver_ranges)

ssource_depths = ssource_depths.flatten()
rreceiver_ranges = rreceiver_ranges.flatten()

In [ ]:
rays = []
eigs = []

for k in range(len(ssource_depths)):

    source_depth = ssource_depths[k]
    receiver_range = rreceiver_ranges[k]
    source_range = 0
    receiver_depths = [float(Z_lower_array)]
    launch_angles = np.linspace(-35,35,1001)
    num_range_save = 1000

    print(f'Run {k+1}/{len(ssource_depths)}, zs = {source_depth}[m], r={receiver_range/1000} [km]...')
    
    rays.append(pr.shoot_rays(source_depth, source_range, launch_angles, receiver_range, num_range_save, envi))
    eigs.append(pr.find_eigenrays(rays[k], receiver_depths, source_depth, source_range, receiver_range, num_range_save, envi))

In [ ]:
import panel as pn
pn.extension()

In [ ]:
def plot_ray_lignup(source_depth_slider, receiver_range_slider):

    source_idx = np.argmin(np.abs(source_depths - source_depth_slider))
    range_idx = np.argmin(np.abs(receiver_ranges - receiver_range_slider*1e3))

    # Calculate the run index from the meshgrid
    run_idx = range_idx * len(source_depths) + source_idx
    
    fig = plt.figure(figsize=(10,4))
    
    t_m6 = eigs[run_idx].ts[0][np.where(eigs[run_idx].ray_id[0]=='-6.0')[0][0],-1]
    DT = t_m6 - 100.75
    
    (20*np.log10(np.abs(mean_pulse_lower)) - mean_pulse_lower_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0)
    (20*np.log10(np.abs(mean_pulse_upper)) - mean_pulse_upper_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0, add_colorbar=False)
    
    plt.scatter(rays[run_idx].ts[:,-1] - DT, -rays[run_idx].zs[:,-1], c='k', s=5, lw=0)
    plt.xlim([100,105])
    plt.ylim([1500,0])
    
    plt.title(f'Source Depth = {ssource_depths[run_idx]} [m], Range = {rreceiver_ranges[run_idx]} [km]\t$\\Delta \\tau$ = {DT:0.3f} [s]')
    #plt.hlines(-1072, 90, 110)
    plt.close()
    return pn.pane.Matplotlib(fig, tight=True)

source_depth_slider = pn.widgets.DiscreteSlider(
    name='Source Depth [m]',
    options=list(source_depths),
    value=source_depths[0]
)

receiver_range_slider = pn.widgets.DiscreteSlider(
    name='Receiver Range [km]',
    options=list(receiver_ranges/1e3),
    value=receiver_ranges[0]/1e3
)

In [ ]:
bound_plot = pn.bind(plot_ray_lignup, source_depth_slider=source_depth_slider, receiver_range_slider=receiver_range_slider)
app = pn.Column(source_depth_slider, receiver_range_slider, bound_plot)
app

### Explore ray variables etc

In [ ]:
ray_run = {
    "rays": rays,
    "eigs": eigs,
    "rays_back": rays_back,
    "eigs_back": eigs_back,
    "envi": envi,
    "envi_back": envi_back,
    "geo": geo,
}

rvs = tools.compute_n23_ray_variables(envi_field = fctd, ray_run=ray_run, ray_ids = ['+7', '-7', '+6', '-6'])

In [ ]:
envi.sound_speed.plot()

plt.plot(rvs['-6'][0]['rs'], -rvs['-6'][0]['zs'])
plt.plot(rvs['+6'][0]['rs'], -rvs['+6'][0]['zs'])

plt.xlim([0,153000])
plt.ylim([5000,0])

ax2 = plt.twinx()

plt.plot(rvs['-6'][0]['rs'], rvs['-6'][0]['RT_zeta_1'])
plt.plot(rvs['+6'][0]['rs'], rvs['+6'][0]['RT_zeta_1'])

plt.ylim([-1, 3.5])

In [ ]:
fctd['N'].plot()

In [ ]:
rvs['+6'][0]['Nray'].plot()
rvs['-6'][0]['Nray'].plot()

In [ ]:
sns.set_palette('tab10')
tools.plot_ray_variables(rvs, ['+7', '-7', '+6', '-6'])

In [ ]:
#plt.plot(rvs['+7'][0]['RT_zeta_1'])
#plt.plot(rvs['+7'][1]['RT_zeta_1'], c='C0')
#plt.plot(rvs['-7'][0]['RT_zeta_1'])
sns.set_palette('tab20')

plt.plot(rvs['+6'][0]['RT_zeta_1'])
plt.plot(rvs['+6'][0]['RT_zeta_2'])

plt.plot(rvs['-6'][0]['RT_zeta_1'])
plt.plot(rvs['-6'][0]['RT_zeta_2'])

In [ ]:
plt.plot(rvs['+6'][0]['Rf2'])
plt.plot(rvs['-6'][0]['Rf2'])

In [ ]:
dzr = np.linspace(0,300,150)
Ds = {}
for ray_id in ['+7','-7', '+6', '-6']:
    Ds[ray_id] = tools.compute_phase_structure_function(rvs[ray_id][0], dzr)

In [ ]:
for ray_id in ['+7','-7', '+6', '-6']:
    plt.plot(dzr, np.exp(-Ds[ray_id]/2), label=ray_id)
plt.xlabel('$\\Delta z$ [m]')

plt.legend()

In [ ]:
for ray_id in ['+7','-7', '+6', '-6']:
    plt.plot(dzr, np.exp(-Ds[ray_id]/2))
plt.xlabel('$\\Delta z$ [m]')

In [ ]:
Ds2 = tools.compute_phase_structure_function(rvs['-5b'][1], dzr)

In [ ]:
len(rvs['-5b'])

## Sweep range values archive

In [ ]:
fn = '../auxiliary/mean_pulse_lower.nc'
mean_pulse_lower = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_upper.nc'
mean_pulse_upper = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_lower_noise.nc'
mean_pulse_lower_noise = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_upper_noise.nc'
mean_pulse_upper_noise = xr.open_dataarray(fn)

In [ ]:
rays = []
eigs = []

r_ranges = [148e3, 149e3, 150e3, 151e3, 152e3, 153e3, 154e3, 155e3]

for r_range in r_ranges:
    ray, eig = tools.run_rays(fctd, num_rays = 3000, source_depth=source_depth, receiver_range = r_range, receiver_depths = receiver_depth)

    rays.append(ray)
    eigs.append(eig)

In [ ]:
for r_range in [154e3, 155e3]:
    ray, eig = tools.run_rays(fctd, num_rays = 3000, source_depth=source_depth, receiver_range = r_range, receiver_depths = receiver_depth)

    rays.append(ray)
    eigs.append(eig)

In [ ]:
r_ranges = [148e3, 149e3, 150e3, 151e3, 152e3, 153e3, 154e3, 155e3]

In [ ]:
for run_idx in range(8):

    fig = plt.figure(figsize=(10,4))
    
    t_m6 = eigs[run_idx].ts[0][np.where(eigs[run_idx].ray_id[0]=='-6.0')[0][0],-1]
    DT = t_m6 - 100.725

    
    (20*np.log10(np.abs(mean_pulse_lower)) - mean_pulse_lower_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0)
    (20*np.log10(np.abs(mean_pulse_upper)) - mean_pulse_upper_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0, add_colorbar=False)
    
    plt.scatter(rays[run_idx].ts[:,-1] - DT, -rays[run_idx].zs[:,-1], c='k', s=5, lw=0)
    plt.xlim([100,102])
    plt.ylim([1500,0])
    
    plt.title(f'Range = {r_ranges[run_idx]/1000} [km], \t$\\Delta \\tau$ = {DT:0.3f} [s]')
    #plt.hlines(-1072, 90, 110)

    fig.savefig(f'figures/ray_linup_zoomed_{run_idx:04}.png', dpi=300)
    plt.close()

In [ ]:
for run_idx in range(8):

    fig = plt.figure(figsize=(10,4))
    
    t_m6 = eigs[run_idx].ts[0][np.where(eigs[run_idx].ray_id[0]=='-8.0b')[0][0],-1]
    DT = t_m6 - 102.44

    
    (20*np.log10(np.abs(mean_pulse_lower)) - mean_pulse_lower_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0)
    (20*np.log10(np.abs(mean_pulse_upper)) - mean_pulse_upper_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0, add_colorbar=False)
    
    plt.scatter(rays[run_idx].ts[:,-1] - DT, -rays[run_idx].zs[:,-1], c='k', s=5, lw=0)
    plt.xlim([100, 110])
    plt.ylim([1500,0])
    
    plt.title(f'Range = {r_ranges[run_idx]/1000} [km], \t$\\Delta \\tau$ = {DT:0.3f} [s]')
    
    fig.savefig(f'figures/ray_linup{run_idx:04}.png', dpi=300)
    plt.close()

In [ ]:
os.system('file figures/ray_linup_zoomed_0000.png')

In [ ]:
os.system('ffmpeg -r 3 -f image2 -s 3000x1200 -i figures/ray_linup_zoomed_%04d.png -vcodec libx264 -crf 25  -pix_fmt yuv420p -y ray_lineup_zoomed.mp4')

In [ ]:
os.system('ffmpeg -r 3 -f image2 -s 3000x1200 -i figures/ray_linup%04d.png -vcodec libx264 -crf 25  -pix_fmt yuv420p -y ray_linup.mp4')

 ## Sweep depth values archive

In [ ]:
fn = '../auxiliary/mean_pulse_lower.nc'
mean_pulse_lower = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_upper.nc'
mean_pulse_upper = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_lower_noise.nc'
mean_pulse_lower_noise = xr.open_dataarray(fn)

fn = '../auxiliary/mean_pulse_upper_noise.nc'
mean_pulse_upper_noise = xr.open_dataarray(fn)

In [ ]:
rays = []
eigs = []

z_ss = [2100, 2150, 2200, 2250, 2300, 2350]

for z_s in z_ss:
    ray, eig = tools.run_rays(fctd, num_rays = 3000, source_depth=z_s, receiver_range = 149511.11757364927, receiver_depths = receiver_depth)

    rays.append(ray)
    eigs.append(eig)

In [ ]:
for run_idx in range(6):

    fig = plt.figure(figsize=(10,4))
    
    t_m6 = eigs[run_idx].ts[0][np.where(eigs[run_idx].ray_id[0]=='-6.0')[0][0],-1]
    DT = t_m6 - 100.725

    
    (20*np.log10(np.abs(mean_pulse_lower)) - mean_pulse_lower_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0)
    (20*np.log10(np.abs(mean_pulse_upper)) - mean_pulse_upper_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0, add_colorbar=False)
    
    plt.scatter(rays[run_idx].ts[:,-1] - DT, -rays[run_idx].zs[:,-1], c='k', s=5, lw=0)
    plt.xlim([100,102])
    plt.ylim([1500,0])
    
    plt.title(f'Source Depth = {z_ss[run_idx]} [m], \t$\\Delta \\tau$ = {DT:0.3f} [s]')
    #plt.hlines(-1072, 90, 110)
    
    fig.savefig(f'figures/source_depth_sweep_ray_linup_zoomed_{run_idx:04}.png', dpi=300)
    plt.close()

In [ ]:
os.system('ffmpeg -r 3 -f image2 -s 3000x1200 -i figures/source_depth_sweep_ray_linup_zoomed_%04d.png -vcodec libx264 -crf 25  -pix_fmt yuv420p -y ray_linup_depth_sweep.mp4')

In [ ]:
for run_idx in range(8):

    fig = plt.figure(figsize=(10,4))
    
    t_m6 = eigs[run_idx].ts[0][np.where(eigs[run_idx].ray_id[0]=='-8.0b')[0][0],-1]
    DT = t_m6 - 102.44

    
    (20*np.log10(np.abs(mean_pulse_lower)) - mean_pulse_lower_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0)
    (20*np.log10(np.abs(mean_pulse_upper)) - mean_pulse_upper_noise).plot(x='time', cmap=cmps.plasma_coolwarm(), vmin=0, add_colorbar=False)
    
    plt.scatter(rays[run_idx].ts[:,-1] - DT, -rays[run_idx].zs[:,-1], c='k', s=5, lw=0)
    plt.xlim([100, 110])
    plt.ylim([1500,0])
    
    plt.title(f'Range = {r_ranges[run_idx]/1000} [km], \t$\\Delta \\tau$ = {DT:0.3f} [s]')
    
    fig.savefig(f'figures/ray_linup{run_idx:04}.png', dpi=300)
    plt.close()